# KNN / WKNN trên UJIIndoorLoc

So sánh **kNN**, **WKNN (1/d)** và **WKNN (1/d²)** theo split của repo, trình bày như notebook Lần 4: Phần A (chỉ số), B (bảng), C (biểu đồ), D (so sánh + kiểm định + kết luận), E (lưu kết quả). Repo private: dùng Colab Secret `GITHUB_TOKEN` có quyền **Contents: Read**, hoặc upload ZIP của nhánh. Không ghi token vào notebook hay output. Để báo cáo kết quả chính thức, chọn full; smoke chỉ kiểm tra luồng.

In [ ]:
# 1. Tải repo private. Sau khi có commit mới, chạy lại ô này.
import io
import shutil
import subprocess
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "feature/knn-wknn"  # sau khi PR được merge có thể đổi thành main
ZIP_PATH = None  # hoặc đường dẫn ZIP đã upload, ví dụ "/content/ips-wifi-research.zip"

repo_zip = Path(ZIP_PATH).read_bytes() if ZIP_PATH else None
if repo_zip is None:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        try:
            request = urllib.request.Request(
                f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}",
                headers={"Authorization": f"Bearer {str(token).strip()}",
                         "Accept": "application/vnd.github+json"})
            with urllib.request.urlopen(request, timeout=120) as response:
                repo_zip = response.read()
        except urllib.error.HTTPError as error:
            print(f"GitHub trả HTTP {error.code}. Kiểm tra token có quyền Contents: Read,"
                  " đã chọn đúng repo, bật quyền truy cập Secret cho notebook và BRANCH tồn tại.")
        except urllib.error.URLError:
            print("Không kết nối được GitHub API. Có thể dùng ZIP của repo để tiếp tục.")
    token = None
if repo_zip is None:
    print("Upload ZIP của đúng nhánh repo từ GitHub (Code → Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP")
    repo_zip = next(iter(uploaded.values()))

root = Path("/content/knn_repo_source")
if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as archive:
    archive.extractall(root)
projects = list(root.rglob("pyproject.toml"))
if len(projects) != 1:
    raise RuntimeError(f"ZIP cần đúng một pyproject.toml; tìm thấy {len(projects)}")
project = projects[0].parent
if not (project / "src/ips/neighbors").is_dir():
    raise RuntimeError("Nhánh/ZIP này chưa có KNN. Chọn BRANCH=feature/knn-wknn hoặc ZIP của nhánh KNN.")
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)
subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--list-options"], check=True)


In [ ]:
# 2. Tải UJIIndoorLoc từ UCI nếu chưa có hai CSV.
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name} trong ZIP UJI")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


In [ ]:
# 3. Chạy: chọn model và cấu hình (smoke kiểm tra luồng, full dùng toàn bộ dữ liệu). Kết quả hiển thị ở các phần B, C, D bên dưới.
import sys
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import Image, display
from IPython.utils.capture import capture_output

sys.path.insert(0, str(project / 'src'))  # để import ips ngay trong phiên Colab
from ips.neighbors import available_models, build_model

MODELS = sorted(available_models())  # Ví dụ chỉ chạy một: ['knn']
CONFIG = "configs/knn_smoke.json"  # đổi thành configs/knn_full.json khi sẵn sàng
settings = json.loads((project / CONFIG).read_text(encoding='utf-8'))
if not MODELS or any(m not in available_models() for m in MODELS):
    raise ValueError('MODELS phải là các model trong danh sách khả dụng')
settings['models'] = MODELS
selected_config = Path('/content/selected_knn_config.json')
selected_config.write_text(json.dumps(settings, indent=2), encoding='utf-8')
NAME = {m: build_model(m).display_name for m in MODELS}
print('Cấu hình:', CONFIG, '| model:', MODELS, '| k_max:', settings['k_max'], '| seed:', settings['seed'], '| val_size:', settings['val_size'])
print('Số dòng dữ liệu:', {p.name: sum(1 for _ in p.open()) - 1 for p in data_dir.glob('*.csv')})
run_dir = Path("/content/runs") / datetime.now(timezone.utc).strftime("knn_%Y%m%d_%H%M%S_%f")
result = subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                        "--data-dir", str(data_dir), "--output-dir", str(run_dir)],
                       capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("knn_cli lỗi - xem thông báo ở trên")
lines = result.stdout.splitlines()
(run_dir / "console_log.txt").write_text(result.stdout, encoding="utf-8")
print("\n".join(line for line in lines if line.startswith("[tòa")))

# Đọc kết quả để các phần B, C, D bên dưới hiển thị
sweep = pd.read_csv(run_dir / "sweep_by_k.csv")
summary_table = pd.read_csv(run_dir / "summary_table.csv")
d1 = pd.read_csv(run_dir / "pairwise_same_k.csv", index_col=0)
d2 = pd.read_csv(run_dir / "wilcoxon.csv")
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
start = next((i for i, line in enumerate(lines) if line.startswith("=" * 40)), None)
end = next((i for i, line in enumerate(lines) if line.startswith("Lưu ý")), None)
conclusion = lines[start:end + 1] if start is not None and end is not None else []


def show_chart(name):
    path = run_dir / name
    if path.exists():
        display(Image(filename=str(path), width=820))
    else:
        print(f"Chưa có {name} (có thể thiếu matplotlib khi chạy).")


extra_error = None
with capture_output() as extra:  # phụ lục của repo (sai số theo tòa, ma trận nhầm lẫn tầng); hiện ở phần F
    try:
        from ips.experiments.visualize import render_run
        report_table, report_figures = render_run(run_dir)
    except Exception as error:
        extra_error = error
if extra_error is not None:
    print("Bỏ qua phụ lục (render_run lỗi):", extra_error)

n_test = next(iter(summary["official_test"].values()))["n"]
print(f"\nĐã chạy xong: {len(MODELS)} model × {sweep.k.nunique()} giá trị k = {len(sweep)} lần đánh giá trên {n_test} mẫu test")
print("k chọn trên validation (sai số trung bình thấp nhất): " + " | ".join(f"{NAME[m]} k={summary['selected_k'][m]}" for m in MODELS))

# PHẦN A — Các chỉ số trong kết quả nghĩa là gì?

Mỗi mẫu test là một lần đo Wi-Fi tại một vị trí đã biết thật. Mô hình đoán **tầng** và **tọa độ (x, y)** (tòa nhà đã biết trước), rồi so với vị trí thật.

| Chỉ số | Ý nghĩa |
|---|---|
| Floor accuracy (%) | Tỉ lệ mẫu đoán đúng tầng. |
| Mean / Median / P90 (m) | Sai số vị trí trung bình / trung vị / phân vị 90, tính trên **tất cả** mẫu test. Nhỏ hơn là tốt hơn. |
| ≤1 m, ≤3 m, ≤5 m (%) | Tỉ lệ mẫu có sai số nằm trong ngưỡng đó. Lớn hơn là tốt hơn. |
| Validation mean (m) | Sai số trên tập validation nội bộ, dùng **để chọn k**. |
| Thời gian | Gộp tọa độ là phần duy nhất ba phương pháp khác nhau; Tổng gồm cả bước tìm láng giềng dùng chung. |

**Lưu ý giao thức (theo repo):** biết sẵn tòa nhà; k được chọn trên validation nội bộ, tập test chỉ dùng để báo cáo. Vì vote tầng giống hệt nhau nên floor accuracy **bằng nhau** giữa ba phương pháp ở cùng k. Giao thức này khác Colab Lần 4 (tham chiếu 100% train, tự vote tòa, chọn k trên test) nên số không so trực tiếp được.

# PHẦN B — Kết quả dạng bảng

### B1. Độ chính xác tầng (%) theo k
Tỉ lệ (%) mẫu test đoán đúng tầng. Các cột giống nhau là bình thường (xem lưu ý ở Phần A).

In [ ]:
t_floor = sweep.pivot(index="k", columns="method", values="floor_acc")[MODELS].rename(columns=NAME)
display(t_floor.round(2))

### B2. Sai số vị trí theo k (mét) — official test
Mean error, Median và P90 của ba phương pháp, tính trên tất cả mẫu test. Số nhỏ hơn là tốt hơn.

In [ ]:
t_err = sweep.pivot(index="k", columns="method", values=["test_mean", "test_median", "test_p90"])
label = {"test_mean": "Mean", "test_median": "Median", "test_p90": "P90"}
t_err.columns = pd.MultiIndex.from_tuples([(label[metric], NAME[m]) for metric, m in t_err.columns])
display(t_err[[(label[c], NAME[m]) for c in ("test_mean", "test_median", "test_p90") for m in MODELS]].round(3))

### B2b. Sai số trung bình theo k (mét) — validation nội bộ
Đây là bảng dùng **để chọn k** (k có sai số nhỏ nhất, hòa thì chọn k nhỏ hơn).

In [ ]:
t_val = sweep.pivot(index="k", columns="method", values="val_mean")[MODELS].rename(columns=NAME)
display(t_val.round(3))
print("k chọn:", " | ".join(f"{NAME[m]} k={summary['selected_k'][m]}" for m in MODELS))

### B3. Thời gian theo k (giây, cho toàn bộ tập test)
`Gộp tọa độ` là phần duy nhất các phương pháp khác nhau; `Tổng` gồm cả bước tìm láng giềng dùng chung.

In [ ]:
t_time = sweep.pivot(index="k", columns="method", values=["t_predict_s", "total_s"])
tl = {"t_predict_s": "Gộp tọa độ", "total_s": "Tổng"}
t_time.columns = pd.MultiIndex.from_tuples([(tl[metric], NAME[m]) for metric, m in t_time.columns])
display(t_time[[(tl[c], NAME[m]) for c in ("t_predict_s", "total_s") for m in MODELS]].round(4))
print(f"Thời gian tìm láng giềng (dùng chung cho cả ba, đo một lần): {summary['t_search_test_s']:.2f} s cho {n_test} mẫu test")

### B4. Bảng tổng hợp tại các k quan trọng
Gồm: k chọn trên validation của từng phương pháp, cả ba ở cùng một k (để so công bằng) và k = 13 (giá trị tham chiếu).

In [ ]:
display(summary_table.round(3))

# PHẦN C — Biểu đồ
Mỗi biểu đồ nằm trong một ô riêng, có chú thích cách đọc phía trên. Màu **cam = kNN**, **xanh dương = WKNN (1/d)**, **xanh lá = WKNN (1/d²)**.

### C1. Độ chính xác tầng theo k
**Cách đọc:** trục ngang là k, trục dọc là % mẫu đoán đúng tầng. Ba đường **trùng nhau** vì vote tầng giống hệt nhau (không phải lỗi).

In [ ]:
show_chart('C1_floor_accuracy.png')

### C2. Sai số vị trí trung bình theo k (official test)
**Cách đọc:** đường **thấp hơn = chính xác hơn**. Đường đứt nét đánh dấu k được chọn trên validation. Ở k = 1 ba đường trùng nhau (chỉ có một láng giềng nên trọng số không còn ý nghĩa).

In [ ]:
show_chart('C2_test_mean_error.png')

### C3. Sai số trung bình theo k (validation nội bộ)
**Cách đọc:** như C2 nhưng trên tập validation, là đường dùng để chọn k. Nếu C2 và C3 cho cùng xu hướng thì việc chọn k trên validation đáng tin hơn.

In [ ]:
show_chart('C3_val_mean_error.png')

### C4. Phân bố sai số (CDF) tại k chọn của mỗi phương pháp
**Cách đọc:** tại một điểm x trên trục ngang, chiều cao của đường là tỉ lệ mẫu có sai số ≤ x mét. **Đường nằm cao hơn / lệch trái hơn = tốt hơn.**

In [ ]:
show_chart('C4_cdf.png')

### C5. Thời gian chạy theo k
**Cách đọc:** đường thấp hơn = nhanh hơn. Thời gian tổng gồm bước tìm láng giềng (chiếm gần hết, giống nhau ở cả ba) nên các đường thường rất sát nhau.

In [ ]:
show_chart('C5_time.png')

### C6. Chênh lệch sai số giữa các cặp phương pháp ở từng k
**Cách đọc:** mỗi cột là `sai số phương pháp A − sai số phương pháp B` ở một giá trị k. **Cột dương = B chính xác hơn A** ở k đó, cột âm = B kém hơn.

In [ ]:
show_chart('C6_chenh_lech.png')

# PHẦN D — Đánh giá: các phương pháp khác nhau thế nào?

### D1. So sánh ở cùng k (k = 1 → k_max)
Bảng hiệu số sai số trung bình (mét) của từng cặp ở mỗi k, **không phụ thuộc** việc chọn k. Giá trị **dương = phương pháp thứ hai trong cặp chính xác hơn**.

In [ ]:
display(d1.round(4))
n_ge2 = len(d1) - 1
for col in d1.columns:
    v = d1[col].iloc[1:]
    second = col.split(" − ")[1]
    print(f"{col} (k = 2..{d1.index.max()}): {second} tốt hơn ở {int((v > 1e-9).sum())}/{n_ge2} giá trị k, "
          f"kém hơn ở {int((v < -1e-9).sum())}/{n_ge2}; chênh trung bình {v.mean():+.3f} m")

### D2. So sánh từng mẫu + kiểm định thống kê (Wilcoxon cặp)
Với cùng một mẫu test, xem phương pháp nào cho sai số nhỏ hơn, rồi dùng **Wilcoxon cặp** để biết chênh lệch có ý nghĩa thống kê hay chỉ do ngẫu nhiên. Ngưỡng dùng hiệu chỉnh Bonferroni (0,05 / số cặp). **Có ý nghĩa thống kê chưa chắc có ý nghĩa thực tế**: hãy nhìn cả cột chênh lệch (m).

In [ ]:
alpha = 0.05 / d2.groupby("k").size().iloc[0]
cols = {"cap": "Cặp (A − B)", "mean_a": "Mean A (m)", "mean_b": "Mean B (m)", "diff_a_minus_b": "A − B (m)",
        "pct_b_better": "% mẫu B nhỏ hơn", "pct_a_better": "% mẫu A nhỏ hơn", "pct_tie": "% bằng nhau",
        "p_value": "p (Wilcoxon)", "significant": "Có ý nghĩa"}
for k_cmp, group in d2.groupby("k"):
    print(f"\n=== Tại k = {k_cmp} ({n_test} mẫu test; ngưỡng Bonferroni {alpha:.4f}) ===")
    shown = group[list(cols)].rename(columns=cols).reset_index(drop=True)
    display(shown.style.format({"Mean A (m)": "{:.3f}", "Mean B (m)": "{:.3f}", "A − B (m)": "{:+.3f}",
                                "% mẫu B nhỏ hơn": "{:.1f}", "% mẫu A nhỏ hơn": "{:.1f}", "% bằng nhau": "{:.1f}",
                                "p (Wilcoxon)": "{:.3g}"}).hide(axis="index"))

### D3. Kết luận tự động
Tóm tắt ba tiêu chí: độ chính xác tầng, sai số vị trí (k chọn trên validation và cùng k), thời gian.

In [ ]:
print("\n".join(conclusion) if conclusion else "Không tìm thấy khung kết luận; xem console_log.txt trong thư mục kết quả.")

# PHỤ LỤC — Biểu đồ bổ sung của repo
Sai số theo từng tòa và ma trận nhầm lẫn **tầng** (do `render_run` của repo vẽ).

In [ ]:
extra.show()

# PHẦN E — Lưu kết quả
Nén toàn bộ thư mục kết quả (các bảng CSV, `summary.json`, log đầy đủ và biểu đồ PNG) để tải về.

In [ ]:
# 4. Tải kết quả về máy.
archive_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive_path)
files.download(archive_path)
